# Phase 7: Isolation Forest Anomaly Detection and Safety Screening
## AI-Driven Intelligent Greywater Management and Smart Reuse Routing System

### Section 1: Objective & System Concept
The objective of Phase 7 is to implement an independent, unsupervised anomaly-detection layer using **Isolation Forest**.
The purpose is **NOT** to replace the supervised XGBoost routing model, but to screen for rare, out-of-distribution, or potentially hazardous greywater batches that warrant an automated safety override or operator review.

```text
Water-Quality Telemetry
        ↓
Isolation Forest Anomaly Screening
        ↓
Statistical Status: Normal / Anomaly
        ↓
Supervised XGBoost Routing Prediction
        ↓
Context-Aware Safety Override Arbitrator
        ↓
Final Routing Decision
```

> **Critical Operational Distinction:**  
> **ANOMALY $\neq$ AUTOMATIC PROOF OF HAZARD.**  
> An anomaly signifies statistical divergence from reference baseline data. A benign statistical oddity triggers review, whereas an anomaly verified by independent physical hazard exceedances triggers an automated Sewer Bypass override.


### Section 2: Load Training, Validation, and Test Data
Load the Phase 4 preprocessed splits. Isolation Forest is fitted **strictly** on the training set.


In [ ]:
import os
import sys
import types
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root
sys.path.insert(0, os.path.abspath('..'))

from anomaly.anomaly_detector import (
    GreywaterAnomalyDetector,
    check_severe_conditions,
    apply_safety_override,
    EXPECTED_FEATURES,
    SEVERE_THRESHOLDS
)

train_df = pd.read_csv('../dataset/processed/train.csv')
val_df = pd.read_csv('../dataset/processed/validation.csv')
test_df = pd.read_csv('../dataset/processed/test.csv')

meta_val = pd.read_csv('../dataset/processed/metadata_val.csv')
meta_test = pd.read_csv('../dataset/processed/metadata_test.csv')

target_col = 'Routing_Class'
feature_cols = [c for c in train_df.columns if c != target_col]

X_train, y_train = train_df[feature_cols].copy(), train_df[target_col].copy()
X_val, y_val = val_df[feature_cols].copy(), val_df[target_col].copy()
X_test, y_test = test_df[feature_cols].copy(), test_df[target_col].copy()

class_names = {
    0: 'Sewer Bypass',
    1: 'Bio-filtration',
    2: 'Restricted Irrigation',
    3: 'Indoor Reuse'
}

print(f"Loaded datasets: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}.")
print(f"Verified {len(feature_cols)} predictor features.")


### Section 3: Feature Preparation & Legitimate Inputs
Confirm that no label leaks, Sample IDs, or post-decision metadata enter the anomaly detector.


In [ ]:
forbidden_cols = [
    'Sample_ID', 'Routing_Class', 'Routing_Class_Name', 'Water_Quality_Class',
    'Primary_Routing_Reason', 'Triggered_Parameters', 'Treatment_Required', 'Safety_Flag'
]
assert not any(c in feature_cols for c in forbidden_cols), "Data leakage detected!"
print("Feature audit passed: Zero target, metadata, or post-decision leakage.")
print("Input Features:", feature_cols)


### Section 4: Isolation Forest Configuration
Load the serialized model trained with `contamination=0.02` (2% statistical tail expectation matching Phase 2 extreme bounds).


In [ ]:
iso_model = joblib.load('../models/isolation_forest.pkl')
with open('../models/isolation_forest_metadata.json', 'r') as f:
    iso_meta = json.load(f)

print("Isolation Forest Configuration:")
for k, v in iso_meta.items():
    if k != 'feature_names':
        print(f"  {k}: {v}")


### Section 5: Contamination Sensitivity Comparison
Compare candidate contamination rates (0.01, 0.02, 0.05, 0.10) on the training set.


In [ ]:
contam_df = pd.read_csv('../reports/isolation_forest_contamination_comparison.csv')
print("Contamination Sensitivity Comparison on Training Set:")
print(contam_df[['contamination_rate', 'anomalies_detected', 'anomaly_pct', 'severe_condition_overlap', 'severe_overlap_pct', 'kitchen_anomalies']].to_string(index=False))


### Section 6: Validation Set Anomaly Analysis
Evaluate anomaly predictions and decision scores on the validation set.


In [ ]:
val_anom_df = pd.read_csv('../reports/isolation_forest_validation_predictions.csv')
print(f"Validation Total Samples: {len(val_anom_df)}")
print(f"Validation Anomalies Detected: {val_anom_df['is_anomaly'].sum()} ({val_anom_df['is_anomaly'].mean()*100:.2f}%)")
print("\nValidation Anomaly Samples:")
print(val_anom_df[val_anom_df['is_anomaly']].to_string(index=False))


### Section 7: Test Set Anomaly Analysis
Inspect anomaly screening on the untouched test set.


In [ ]:
test_anom_df = pd.read_csv('../reports/isolation_forest_test_predictions.csv')
print(f"Test Total Samples: {len(test_anom_df)}")
print(f"Test Anomalies Detected: {test_anom_df['is_anomaly'].sum()} ({test_anom_df['is_anomaly'].mean()*100:.2f}%)")
print("\nTest Anomaly Samples:")
print(test_anom_df[test_anom_df['is_anomaly']].to_string(index=False))


### Section 8: Source-Wise Anomaly Distribution
Analyze how anomalies distribute across the four greywater origins.


In [ ]:
source_df = pd.read_csv('../reports/anomaly_by_source.csv')
print("Anomaly Distribution by Greywater Source:")
print(source_df.to_string(index=False))

plt.figure(figsize=(8, 4))
x = np.arange(len(source_df))
width = 0.35
plt.bar(x - width/2, source_df['normal_samples'], width, label='Normal', color='#3b82f6')
plt.bar(x + width/2, source_df['anomaly_samples'], width, label='Anomaly', color='#ef4444')
plt.xticks(x, source_df['source'])
plt.ylabel('Samples')
plt.title('Isolation Forest Anomalies by Greywater Source')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### Section 9: Water Quality Comparison (Normal vs. Anomaly)
Compare statistical distributions of physicochemical parameters between normal and anomalous samples.


In [ ]:
quality_df = pd.read_csv('../reports/anomaly_quality_comparison.csv')
print("Top Divergent Water Quality Parameters:")
print(quality_df.sort_values(by='mean_difference', key=abs, ascending=False).head(8).to_string(index=False))


### Section 10: Routing Class vs. Anomaly Relationship
Examine empirical overlap between rule-derived routing targets and statistical anomalies.
*Note: This is an exploratory comparison; routing labels are not anomaly ground truth.*


In [ ]:
matrix_df = pd.read_csv('../reports/routing_class_anomaly_matrix.csv')
print("Routing Class vs. Anomaly Matrix:")
print(matrix_df.to_string(index=False))


### Section 11: Anomaly Safety Auditing
Audit evaluation data across the four operational safety quadrants.


In [ ]:
safety_df = pd.read_csv('../reports/anomaly_safety_analysis.csv')
print("Anomaly Safety Auditing:")
print(safety_df[['category', 'sample_count', 'percentage_of_total']].to_string(index=False))


### Section 12: Safety Override Simulation on Test Set
Simulate the conceptual safety arbitration logic combining XGBoost predictions with anomaly status and physical hazard screening.


In [ ]:
xgb_model = joblib.load('../models/xgboost_optimized.pkl')
xgb_test_preds = xgb_model.predict(X_test)

override_statuses = []
final_classes = []
overrides_applied = []

for i in range(len(X_test)):
    row = X_test.iloc[i].to_dict()
    is_anom = bool(test_anom_df.iloc[i]['is_anomaly'])
    score = float(test_anom_df.iloc[i]['anomaly_score'])
    res = apply_safety_override(int(xgb_test_preds[i]), is_anom, score, row)
    override_statuses.append(res['safety_status'])
    final_classes.append(res['final_routing_class'])
    overrides_applied.append(res['override_applied'])

override_summary = pd.Series(override_statuses).value_counts()
print("Safety Status Breakdown on Test Set (225 samples):")
print(override_summary)
print(f"Total Critical Safety Overrides Applied: {sum(overrides_applied)}")


### Section 13: Final Conclusions & Scientific Limitations
1. **Anomaly Screening:** Isolation Forest (`contamination=0.02`) successfully identifies severe multi-dimensional statistical outliers with 100% overlap with extreme physical parameter exceedances in training.
2. **Source Affinity:** Statistical anomalies are concentrated in Kitchen greywater (~10.1%), matching physical expectations of high organic solids and grease loading.
3. **Safety Override Arbitrator:** The dual-gate architecture successfully arbitrates between ML routing and safety screening, applying 8 critical safety overrides on the test set while clearing 170 normal samples.
4. **Mandatory Scientific Limitation:**
   > *"Isolation Forest detects statistical anomalies and does not independently prove that a sample is hazardous. It operates as an exploratory, unsupervised safety screening layer rather than a clinically validated ground-truth classifier."*
